# Whole-exome sequencing: a rare form of familial hypercholesterolemia

**Purpose.** Work through a real diagnostic question: given the whole-exome
sequence of one small family, find the single variant that explains a child's
disease. The exercise is about **filtering** — an exome holds tens of thousands
of variants, and almost all of them are harmless. You will use two filters,
**population frequency** and **predicted functional consequence**, and you will
work out the frequency cutoff yourself from the prevalence of the disease.

**The data.** Human whole-exome sequencing of a family of four: two parents and
two children, one of whom (**HG04204**) has abnormally high cholesterol. The
individual IDs are 1000 Genomes IDs. These are **called genotypes**, already
annotated with the predicted consequence of each variant and with its allele
frequency in **gnomAD**, the largest public catalogue of human variation. The
table has one row per variant and one genotype column per family member.

**The family.** The parents are second cousins, and there is no record of high
cholesterol anywhere else in the family.

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data moves, this is the ONLY cell you need to change.
# No cell below this one uses a full path.
#############################################################
DATA=/course/data/current_data/wes
WES=$DATA/ex02.wes.rds

WORK_DIR=$HOME/wes_fh_human
mkdir -p $WORK_DIR
cd $WORK_DIR

# R cannot read bash variables, so write the paths to a file it can read
cat > $WORK_DIR/env.sh <<EOF
export DATA=$DATA
export WES=$WES
export WORK=$WORK_DIR
EOF

ls -l $DATA/

In [ ]:
# R cannot source env.sh, so read the paths out of it rather than repeating them
env <- readLines(path.expand("~/wes_fh_human/env.sh"))
getvar <- function(k) sub(paste0('^export ', k, '='), '', grep(paste0('^export ', k, '='), env, value = TRUE)[1])
DATA <- getvar("DATA"); WES <- getvar("WES"); WORK <- getvar("WORK")
setwd(WORK)

**Q1. If the abnormal cholesterol level is caused by genetics, what type of inheritance is most likely, given that the parents are second cousins and nobody else in the family is affected?**

*Double-click this text to write your answer*

Now load the whole-exome data for the family.

In [ ]:
# Load the WES data
vars <- readRDS(WES)

consequences <- c('transcript_ablation','splice_acceptor_variant','splice_donor_variant','stop_gained','frameshift_variant','stop_lost','start_lost','transcript_amplification','inframe_insertion','inframe_deletion','missense_variant','protein_altering_variant','splice_region_variant','incomplete_terminal_codon_variant','start_retained_variant','stop_retained_variant','synonymous_variant')

paste0('Loaded ', nrow(vars), ' WES variants')

- How many variants are there in the exome of this family?
- If you tested every one of them for association with the disease in four people, how many would you expect to look convincing by chance?

**Q2. Assuming a prevalence of familial hypercholesterolemia (FH) of 0.1%, what is the maximum allele frequency a high-penetrance *recessive* FH variant could have?**

For a recessive variant, only homozygotes are affected, so

$$Prevalence = P(homozygous) = AF^2$$

Modify the code below until the prevalence it prints matches 0.1%.

In [ ]:
# Chosen allele frequency
AF = 0.032 # <- change this

# Calculate prevalence from chosen allele frequency
prevalence = AF^2

# Print out prevalence in percentage:
paste0('FH prevalence due to this variant = ', prevalence*100, '%')

- What frequency did you end up with?
- The same calculation for a *dominant* variant would use $2AF(1-AF)+AF^2$ instead. Would that give a higher or a lower cutoff, and why?

From [ensembl](https://www.ensembl.org/info/genome/variation/prediction/predicted_data.html) we have the following severity-order of functional consequences

|Order|Consequence|IMPACT|
|---|---|---|
|1|transcript_ablation|HIGH|
|2|splice_acceptor_variant|HIGH|
|3|splice_donor_variant|HIGH|
|4|stop_gained|HIGH|
|5|frameshift_variant|HIGH|
|6|stop_lost|HIGH|
|7|start_lost|HIGH|
|8|transcript_amplification|HIGH|
|9|inframe_insertion|MODERATE|
|10|inframe_deletion|MODERATE|
|11|missense_variant|MODERATE|
|12|protein_altering_variant|MODERATE|
|13|splice_region_variant|LOW|
|14|incomplete_terminal_codon_variant|LOW|
|15|start_retained_variant|LOW|
|16|stop_retained_variant|LOW|
|17|synonymous_variant|LOW|

The list is ordered, so "this consequence or worse" is simply the start of the list
up to the one you name.

Now filter the variants. Two things to set:

1. The **worst consequence** you will allow — start at `stop_gained`, which truncates the protein.
2. The **maximum gnomAD allele frequency** — use the number you worked out in **Q2**.

In [ ]:
# Filtering
# Try to change the maximum gnomAD AF and the worst consequence.
max_gnomad_AF = 0.033
consequence_or_worse_than = 'stop_gained'

or_worse = consequences[1:which(consequences == consequence_or_worse_than)]
subset(vars,
       gnomAD_freq < max_gnomad_AF &
       Consequence %in% or_worse)

- How many variants survive the filter?
- Try loosening the consequence filter to `missense_variant`. How many more do you get, and is the candidate still obvious?

To find a candidate variant, try

- looking at the genotype of the affected individual (**HG04204**) — under the inheritance model from **Q1**, what should it be?
- googling the gene name of each candidate to find out what the gene does
- setting a stricter maximum gnomAD frequency

**Q3. Propose a candidate disease variant that follows the inheritance model from Q1.**

*Double-click this text to write your answer*

# Continue with the next exercise
`wes_diabetes_human.ipynb`

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/gwas/quiz/wes_fh.json")